In [1]:
# =================================================================================================
# NHANES 2005-2006
# Sleep Disorders Questionnaire
#
# File:
#   SLQ_D.xpt
#
# Purpose:
#   Convert XPT -> CSV while preserving NHANES fidelity.
#
# Fidelity rules:
#   - Remove SEQN ".0"
#   - Restore ONLY validated XPORT tiny-zero artifacts
#   - Preserve missing positions
#   - Do not round source values
#   - Do not use float_format=
#   - Verify CDC frequencies
#   - Verify exact XPT -> CSV round-trip
#
# NOTE:
#   SLQ_D contains no genuine decimal measurement variables.
#   All 28 released fields are whole-number IDs / codes /
#   hours / minutes after validated zero restoration.
# =================================================================================================


# =================================================================================================
# 1. IMPORTS
# =================================================================================================

import pandas as pd
import numpy as np
from pathlib import Path


# =================================================================================================
# 2. PATHS
# =================================================================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2005_2006"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "SLQ_D.xpt"
csv_file = DATA_DIR / "SLQ_D_corrected.csv"


# =================================================================================================
# 3. READ XPT
# =================================================================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Untouched pandas-decoded source
#
source_df = df.copy()


print("=" * 100)
print("NHANES 2005-2006 SLQ_D XPT -> CSV")
print("=" * 100)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# =================================================================================================
# 4. EXPECTED STRUCTURE
# =================================================================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "SLD010H",
    "SLD020M",
    "SLQ030",
    "SLQ040",
    "SLQ050",
    "SLQ060",
    "SLQ070A",
    "SLQ070B",
    "SLQ070C",
    "SLQ070D",
    "SLQ080",
    "SLQ090",
    "SLQ100",
    "SLQ110",
    "SLQ120",
    "SLQ130",
    "SLQ140",
    "SLQ150",
    "SLQ160",
    "SLQ170",
    "SLQ180",
    "SLQ190",
    "SLQ200",
    "SLQ210",
    "SLQ220",
    "SLQ230",
    "SLQ240"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "SLQ_D columns/order differ from expected.\n\n"
        f"Expected:\n{EXPECTED_COLUMNS}\n\n"
        f"Actual:\n{df.columns.tolist()}"
    )


if len(df) != 6139:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 28:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 6,139 rows x 28 columns."
)


# =================================================================================================
# 5. FIELD TYPE PLAN
# =================================================================================================

#
# Direct inspection shows NO genuine fractional fields.
#
# SLD010H = sleep hours
# SLD020M = minutes to fall asleep
#
# Both are released as whole-number fields.
#
# The remaining SLQ variables are categorical codes.
#

INTEGER_COLUMNS = EXPECTED_COLUMNS.copy()

DECIMAL_COLUMNS = []


print(
    "\n--- FIELD TYPE PLAN ---"
)

print(
    "Whole-number/code fields:",
    len(INTEGER_COLUMNS)
)

print(
    "Genuine decimal fields:",
    DECIMAL_COLUMNS
)


# =================================================================================================
# 6. VERIFY ALL VARIABLES LOADED NUMERICALLY
# =================================================================================================

numeric_columns = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


if numeric_columns != EXPECTED_COLUMNS:

    raise ValueError(
        "Expected all 28 SLQ_D variables "
        "to load numerically."
    )


print(
    "PASS: All 28 variables loaded numerically."
)


# =================================================================================================
# 7. KNOWN SAS XPORT TINY-ZERO ARTIFACT
# =================================================================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


tiny_counts = (
    df[EXPECTED_COLUMNS]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


ordinary_zero_counts = (
    df[EXPECTED_COLUMNS]
    .eq(0)
    .sum()
)


ordinary_zero_total = int(
    ordinary_zero_counts.sum()
)


print(
    "\n--- ORIGINAL XPT ZERO CHECK ---"
)

print(
    "Ordinary numeric zeros before repair:",
    ordinary_zero_total
)

print(
    "Tiny XPORT values:",
    f"{tiny_total:,}"
)


print(
    "\nTiny values by variable:"
)

print(
    tiny_counts[
        tiny_counts > 0
    ]
    .to_string()
)


# =================================================================================================
# 8. VALIDATED SLQ_D TINY-ZERO PATTERN
# =================================================================================================

#
# Directly inspected from attached SLQ_D.xpt.
#
# CDC confirms zero is valid for each field:
#
# SLD020M:
#     minutes-to-sleep range begins at 0
#
# SLQ030 / SLQ040:
#     0 = Never
#
# SLQ080 through SLQ160:
#     0 = Never
#

EXPECTED_TINY_COUNTS = {

    "SLD020M": 97,

    "SLQ030": 2014,
    "SLQ040": 4655,

    "SLQ080": 2629,
    "SLQ090": 2401,
    "SLQ100": 2822,
    "SLQ110": 1918,
    "SLQ120": 2124,
    "SLQ130": 1856,
    "SLQ140": 5190,
    "SLQ150": 4867,
    "SLQ160": 4267
}


actual_tiny_counts = (
    tiny_counts[
        tiny_counts > 0
    ]
    .astype(int)
    .to_dict()
)


if actual_tiny_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "STOP: SLQ_D tiny-value pattern differs "
        "from validated attachment.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{actual_tiny_counts}"
    )


if tiny_total != 34840:

    raise ValueError(
        f"Unexpected total tiny-value count: "
        f"{tiny_total:,}"
    )


if ordinary_zero_total != 0:

    raise ValueError(
        "Unexpected ordinary zeros existed "
        "before XPORT correction."
    )


print(
    "PASS: Exact 34,840-value tiny-zero "
    "pattern validated."
)


# =================================================================================================
# 9. RESTORE ONLY VALIDATED ZEROS
# =================================================================================================

#
# DO NOT use a global:
#
#     df.replace(TINY_VALUE, 0)
#
# Correct only variables already validated.
#

correction_counts = {}


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    mask = (
        df[col]
        .eq(TINY_VALUE)
    )


    actual_count = int(
        mask.sum()
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: tiny count differs.\n"
            f"Expected: {expected_count:,}\n"
            f"Actual:   {actual_count:,}"
        )


    df.loc[
        mask,
        col
    ] = 0


    correction_counts[col] = actual_count


total_corrections = sum(
    correction_counts.values()
)


print(
    "\n--- VALIDATED ZERO RESTORATION ---"
)


for col, count in correction_counts.items():

    print(
        f"{col}: {count:,}"
    )


print(
    "Total restored zeros:",
    f"{total_corrections:,}"
)


if total_corrections != 34840:

    raise ValueError(
        "Total correction count incorrect."
    )


print(
    "PASS: 34,840 validated zeros restored."
)


# =================================================================================================
# 10. EXPECTED CORRECTED SOURCE
# =================================================================================================

expected_df = source_df.copy()


for col in EXPECTED_TINY_COUNTS:

    expected_df.loc[
        expected_df[col]
        .eq(TINY_VALUE),
        col
    ] = 0


# =================================================================================================
# 11. VERIFY NO TINY VALUES REMAIN
# =================================================================================================

remaining_tiny = int(
    df[EXPECTED_COLUMNS]
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print(
    "\nTiny values remaining:",
    remaining_tiny
)


if remaining_tiny != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# =================================================================================================
# 12. FRACTIONAL-VALUE DISCOVERY
# =================================================================================================

#
# Diagnostic only.
#
# np.round() does NOT alter the dataframe here.
#

print(
    "\n--- FRACTIONAL VALUE CHECK ---"
)


fractional_counts = {}


for col in EXPECTED_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        fractional_count = 0

    else:

        fractional_count = int(
            (
                ~np.isclose(
                    values,
                    np.round(values),
                    rtol=0,
                    atol=1e-12
                )
            )
            .sum()
        )


    fractional_counts[col] = (
        fractional_count
    )


    print(
        f"{col}: "
        f"{fractional_count:,} fractional"
    )


fractional_variables = {
    col: count
    for col, count in fractional_counts.items()
    if count > 0
}


if fractional_variables:

    raise ValueError(
        "Unexpected fractional variables found:\n"
        + str(fractional_variables)
    )


print(
    "PASS: No genuine decimal variables in SLQ_D."
)


# =================================================================================================
# 13. SEQN VALIDATION
# =================================================================================================

print(
    "\n--- SEQN ---"
)

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)

print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)

print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if df["SEQN"].min() != 31130:

    raise ValueError(
        "Unexpected minimum SEQN."
    )


if df["SEQN"].max() != 41474:

    raise ValueError(
        "Unexpected maximum SEQN."
    )


if int(
    df["SEQN"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "SEQN contains missing values."
    )


if int(
    df["SEQN"]
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "SEQN contains duplicate values."
    )


print(
    "PASS: SEQN 31130-41474 validated."
)


# =================================================================================================
# 14. FREQUENCY CHECK HELPER
# =================================================================================================

def check_frequency(
    column,
    expected_values,
    expected_missing=0
):

    actual_values = (
        df[column]
        .value_counts()
        .sort_index()
        .to_dict()
    )


    actual_missing = int(
        df[column]
        .isna()
        .sum()
    )


    print(
        f"\n--- CDC CHECK: {column} ---"
    )

    print(
        "Observed:",
        actual_values
    )

    print(
        "Missing:",
        actual_missing
    )


    if actual_values != expected_values:

        raise ValueError(
            f"{column} frequencies differ.\n\n"
            f"Expected:\n{expected_values}\n\n"
            f"Actual:\n{actual_values}"
        )


    if actual_missing != expected_missing:

        raise ValueError(
            f"{column} missing count differs.\n"
            f"Expected: {expected_missing:,}\n"
            f"Actual:   {actual_missing:,}"
        )


    print(
        f"PASS: {column} matches CDC."
    )


# =================================================================================================
# 15. SLD010H - HOURS OF SLEEP
# =================================================================================================

check_frequency(
    "SLD010H",
    {
        1: 9,
        2: 19,
        3: 55,
        4: 208,
        5: 565,
        6: 1324,
        7: 1614,
        8: 1745,
        9: 359,
        10: 154,
        11: 26,
        12: 50,
        77: 1,
        99: 10
    }
)


# =================================================================================================
# 16. SLD020M - MINUTES TO FALL ASLEEP
# =================================================================================================

EXPECTED_SLD020M = {
    0: 97,
    1: 293,
    2: 213,
    3: 92,
    4: 22,
    5: 983,
    6: 7,
    7: 17,
    8: 8,
    9: 4,
    10: 955,
    12: 4,
    13: 2,
    14: 1,
    15: 706,
    17: 1,
    19: 1,
    20: 401,
    21: 1,
    25: 18,
    29: 2,
    30: 1187,
    35: 12,
    39: 1,
    40: 24,
    45: 117,
    50: 1,
    60: 924,
    77: 1,
    99: 44
}


check_frequency(
    "SLD020M",
    EXPECTED_SLD020M
)


# =================================================================================================
# 17. SNORING / BREATHING FREQUENCIES
# =================================================================================================

check_frequency(
    "SLQ030",
    {
        0: 2014,
        1: 1017,
        2: 803,
        3: 1495,
        7: 3,
        9: 807
    }
)


check_frequency(
    "SLQ040",
    {
        0: 4655,
        1: 437,
        2: 274,
        3: 268,
        7: 2,
        9: 503
    }
)


# =================================================================================================
# 18. SLEEP-DIAGNOSIS QUESTIONS
# =================================================================================================

check_frequency(
    "SLQ050",
    {
        1: 1184,
        2: 4946,
        7: 1,
        9: 8
    }
)


check_frequency(
    "SLQ060",
    {
        1: 370,
        2: 5763,
        7: 1,
        9: 5
    }
)


# =================================================================================================
# 19. SLEEP-DISORDER TYPE ROUTING
# =================================================================================================

check_frequency(
    "SLQ070A",
    {
        1: 201,
        9: 28
    },
    expected_missing=5910
)


check_frequency(
    "SLQ070B",
    {
        2: 78
    },
    expected_missing=6061
)


check_frequency(
    "SLQ070C",
    {
        3: 25
    },
    expected_missing=6114
)


check_frequency(
    "SLQ070D",
    {
        4: 62
    },
    expected_missing=6077
)


# =================================================================================================
# 20. SLEEP FREQUENCY QUESTIONS
# =================================================================================================

EXPECTED_SLEEP_FREQUENCIES = {

    "SLQ080": {
        0: 2629,
        1: 1262,
        2: 1304,
        3: 526,
        4: 416,
        7: 1,
        9: 1
    },

    "SLQ090": {
        0: 2401,
        1: 1231,
        2: 1396,
        3: 695,
        4: 409,
        7: 1,
        9: 6
    },

    "SLQ100": {
        0: 2822,
        1: 1112,
        2: 1196,
        3: 625,
        4: 376,
        7: 1,
        9: 7
    },

    "SLQ110": {
        0: 1918,
        1: 996,
        2: 1713,
        3: 932,
        4: 570,
        7: 1,
        9: 9
    },

    "SLQ120": {
        0: 2124,
        1: 1311,
        2: 1599,
        3: 715,
        4: 378,
        7: 1,
        9: 11
    },

    "SLQ130": {
        0: 1856,
        1: 1077,
        2: 1697,
        3: 923,
        4: 564,
        7: 1,
        9: 21
    },

    "SLQ140": {
        0: 5190,
        1: 237,
        2: 277,
        3: 145,
        4: 288,
        7: 1,
        9: 1
    },

    "SLQ150": {
        0: 4867,
        1: 431,
        2: 487,
        3: 192,
        4: 118,
        7: 1,
        9: 43
    },

    "SLQ160": {
        0: 4267,
        1: 731,
        2: 778,
        3: 246,
        4: 109,
        7: 1,
        9: 7
    }
}


for col, expected in (
    EXPECTED_SLEEP_FREQUENCIES.items()
):

    check_frequency(
        col,
        expected
    )


# =================================================================================================
# 21. FUNCTIONAL OUTCOME QUESTIONS
# =================================================================================================

EXPECTED_FUNCTIONAL_FREQUENCIES = {

    "SLQ170": {
        1: 61,
        2: 4539,
        3: 1266,
        4: 207,
        5: 58,
        7: 2,
        9: 6
    },

    "SLQ180": {
        1: 41,
        2: 4895,
        3: 965,
        4: 165,
        5: 59,
        7: 2,
        9: 12
    },

    "SLQ190": {
        1: 27,
        2: 5842,
        3: 218,
        4: 39,
        5: 9,
        7: 2,
        9: 2
    },

    "SLQ200": {
        1: 232,
        2: 5082,
        3: 675,
        4: 108,
        5: 36,
        7: 2,
        9: 4
    },

    "SLQ210": {
        1: 292,
        2: 5087,
        3: 592,
        4: 120,
        5: 42,
        7: 3,
        9: 3
    },

    "SLQ220": {
        1: 305,
        2: 4955,
        3: 685,
        4: 145,
        5: 42,
        7: 3,
        9: 4
    },

    "SLQ230": {
        1: 627,
        2: 4883,
        3: 494,
        4: 101,
        5: 30,
        7: 2,
        9: 2
    },

    "SLQ240": {
        1: 85,
        2: 5390,
        3: 537,
        4: 92,
        5: 32,
        7: 2,
        9: 1
    }
}


for col, expected in (
    EXPECTED_FUNCTIONAL_FREQUENCIES.items()
):

    check_frequency(
        col,
        expected
    )


# =================================================================================================
# 22. SLQ060 -> SLQ070 ROUTING CHECK
# =================================================================================================

#
# SLQ070A-D are only asked after:
#
#     SLQ060 = 1
#
# diagnosed with a sleep disorder.
#
# However, respondents may identify different disorder types,
# and the multi-select variables are not expected to all
# be populated for all 370 participants.
#
# We verify that nobody outside the eligible SLQ060=1
# population has a populated SLQ070 field.
#

sleep_disorder_yes = (
    df["SLQ060"]
    .eq(1)
)


for col in [
    "SLQ070A",
    "SLQ070B",
    "SLQ070C",
    "SLQ070D"
]:

    invalid = int(
        (
            df[col]
            .notna()
            &
            ~sleep_disorder_yes
        )
        .sum()
    )


    print(
        f"{col} outside SLQ060=1:",
        invalid
    )


    if invalid != 0:

        raise ValueError(
            f"{col} contains values outside "
            "the expected SLQ060=1 population."
        )


print(
    "PASS: Sleep-disorder type routing validated."
)


# =================================================================================================
# 23. PRE-CONVERSION EXACT FIDELITY
# =================================================================================================

pre_conversion_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        df[col]
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        pre_conversion_failures.append(
            col
        )


if pre_conversion_failures:

    raise ValueError(
        "Unexpected numeric changes before "
        "Int64 conversion:\n"
        + str(pre_conversion_failures)
    )


print(
    "PASS: Corrected dataframe exactly matches "
    "expected corrected XPT values."
)


# =================================================================================================
# 24. FINAL ZERO PATTERN BEFORE TYPE CONVERSION
# =================================================================================================

final_zero_counts = {}


for col in EXPECTED_COLUMNS:

    count = int(
        df[col]
        .eq(0)
        .sum()
    )


    if count > 0:

        final_zero_counts[col] = count


if final_zero_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "Final zero pattern differs from "
        "validated SLQ_D pattern.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{final_zero_counts}"
    )


print(
    "PASS: Final zero pattern exactly matches "
    "34,840 validated restorations."
)


# =================================================================================================
# 25. CONVERT ALL PROVEN WHOLE-NUMBER FIELDS TO Int64
# =================================================================================================

for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) > 0:

        whole_mask = np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=1e-12
        )


        if not whole_mask.all():

            raise ValueError(
                f"{col} unexpectedly contains "
                "fractional values."
            )


    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "PASS: All 28 SLQ_D fields converted "
    "to nullable Int64."
)


# =================================================================================================
# 26. PRE-EXPORT NUMERIC FIDELITY
# =================================================================================================

pre_export_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        pre_export_failures.append(
            col
        )


if pre_export_failures:

    raise ValueError(
        "Pre-export numeric fidelity failures:\n"
        + str(pre_export_failures)
    )


print(
    "PASS: All 28 variables match expected values."
)


# =================================================================================================
# 27. MISSING-POSITION FIDELITY
# =================================================================================================

missing_errors = 0


for col in EXPECTED_COLUMNS:

    source_missing = (
        source_df[col]
        .isna()
        .to_numpy()
    )


    current_missing = (
        df[col]
        .isna()
        .to_numpy()
    )


    missing_errors += int(
        (
            source_missing
            !=
            current_missing
        )
        .sum()
    )


if missing_errors != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing positions preserved exactly."
)


# =================================================================================================
# 28. EXPORT CSV
# =================================================================================================

#
# No rounding.
# No float_format.
#
# This same export rule remains safe for your broader
# NHANES pipeline where Dietary/Lab/Exam decimals exist.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print(
    "\nCSV created:"
)

print(
    csv_file
)


# =================================================================================================
# 29. RE-READ WITH ROUND-TRIP FLOAT PARSER
# =================================================================================================

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != (
    6139,
    28
):

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV column order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# =================================================================================================
# 30. EXACT XPT -> CSV NUMERIC FIDELITY
# =================================================================================================

roundtrip_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    final_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        final_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        roundtrip_failures.append(
            col
        )


if roundtrip_failures:

    raise ValueError(
        "XPT -> CSV fidelity failures:\n"
        + str(roundtrip_failures)
    )


print(
    "PASS: All 28 variables round-trip exactly."
)


# =================================================================================================
# 31. READ FINAL CSV AS RAW TEXT
# =================================================================================================

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


# =================================================================================================
# 32. ".0" CHECK
# =================================================================================================

dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = raw_csv_df[col]


    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )


    count = int(
        bad_mask.sum()
    )


    if count > 0:

        dot_zero_errors[col] = count


if dot_zero_errors:

    raise ValueError(
        "Unexpected '.0' formatting found:\n"
        + str(dot_zero_errors)
    )


print(
    "PASS: No unnecessary '.0' values."
)


# =================================================================================================
# 33. FINAL TINY-VALUE CHECK
# =================================================================================================

tiny_csv_total = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    tiny_csv_total += int(
        values
        .eq(TINY_VALUE)
        .sum()
    )


if tiny_csv_total != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain in final CSV."
    )


print(
    "PASS: No tiny XPORT values remain."
)


# =================================================================================================
# 34. FINAL ZERO PATTERN
# =================================================================================================

csv_zero_counts = {}


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    zero_count = int(
        values
        .eq(0)
        .sum()
    )


    if zero_count > 0:

        csv_zero_counts[col] = zero_count


if csv_zero_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "Final CSV zero pattern differs.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{csv_zero_counts}"
    )


print(
    "PASS: Final CSV contains exactly "
    "34,840 validated zeros."
)


# =================================================================================================
# 35. SCIENTIFIC NOTATION CHECK
# =================================================================================================

scientific_errors = {}


for col in EXPECTED_COLUMNS:

    count = int(
        raw_csv_df[col]
        .str.contains(
            r"[eE][+-]\d+",
            regex=True
        )
        .sum()
    )


    if count > 0:

        scientific_errors[col] = count


if scientific_errors:

    raise ValueError(
        "Unexpected scientific notation found:\n"
        + str(scientific_errors)
    )


print(
    "PASS: No scientific notation."
)


# =================================================================================================
# 36. FINAL SEQN CHECK
# =================================================================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if final_seqn.min() != 31130:

    raise ValueError(
        "Final minimum SEQN incorrect."
    )


if final_seqn.max() != 41474:

    raise ValueError(
        "Final maximum SEQN incorrect."
    )


if int(
    final_seqn.isna()
    .sum()
) != 0:

    raise ValueError(
        "Final CSV contains missing SEQN."
    )


if int(
    final_seqn.duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Final CSV contains duplicate SEQN."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final SEQN still contains '.0'."
    )


print(
    "PASS: Final SEQN is valid and has no '.0'."
)


# =================================================================================================
# 37. FINAL SUMMARY
# =================================================================================================

print(
    "\n"
    + "=" * 100
)

print(
    "SLQ_D CONVERSION COMPLETE"
)

print(
    "=" * 100
)

print(
    "Cycle: 2005-2006"
)

print(
    "Component: Questionnaire - Sleep Disorders"
)

print(
    "Eligible population: age 16+"
)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)

print(
    "SEQN range:",
    f"{df['SEQN'].min()}-{df['SEQN'].max()}"
)

print(
    "Original tiny XPORT artifacts:",
    f"{tiny_total:,}"
)

print(
    "Variables containing tiny artifacts:",
    len(EXPECTED_TINY_COUNTS)
)

print(
    "Validated zero restorations:",
    f"{total_corrections:,}"
)

print(
    "Genuine decimal variables:",
    DECIMAL_COLUMNS
)

print(
    "Whole-number/code fields:",
    len(INTEGER_COLUMNS)
)

print(
    "Sleep hours SLD010H nonmissing:",
    int(
        df["SLD010H"]
        .notna()
        .sum()
    )
)

print(
    "Sleep latency SLD020M nonmissing:",
    int(
        df["SLD020M"]
        .notna()
        .sum()
    )
)

print(
    "Sleep disorder diagnosis SLQ060=1:",
    int(
        df["SLQ060"]
        .eq(1)
        .sum()
    )
)

print(
    "Exact round-trip failures: 0 expected"
)

print(
    "Missing-position changes: 0 expected"
)

print(
    "Tiny values remaining: 0 expected"
)

print(
    "Unnecessary '.0': 0 expected"
)

print(
    "Scientific notation: 0 expected"
)

print(
    "=" * 100
)

NHANES 2005-2006 SLQ_D XPT -> CSV
Rows: 6,139
Columns: 28
PASS: Structure = 6,139 rows x 28 columns.

--- FIELD TYPE PLAN ---
Whole-number/code fields: 28
Genuine decimal fields: []
PASS: All 28 variables loaded numerically.

--- ORIGINAL XPT ZERO CHECK ---
Ordinary numeric zeros before repair: 0
Tiny XPORT values: 34,840

Tiny values by variable:
SLD020M      97
SLQ030     2014
SLQ040     4655
SLQ080     2629
SLQ090     2401
SLQ100     2822
SLQ110     1918
SLQ120     2124
SLQ130     1856
SLQ140     5190
SLQ150     4867
SLQ160     4267
PASS: Exact 34,840-value tiny-zero pattern validated.

--- VALIDATED ZERO RESTORATION ---
SLD020M: 97
SLQ030: 2,014
SLQ040: 4,655
SLQ080: 2,629
SLQ090: 2,401
SLQ100: 2,822
SLQ110: 1,918
SLQ120: 2,124
SLQ130: 1,856
SLQ140: 5,190
SLQ150: 4,867
SLQ160: 4,267
Total restored zeros: 34,840
PASS: 34,840 validated zeros restored.

Tiny values remaining: 0
PASS: No tiny XPORT artifacts remain.

--- FRACTIONAL VALUE CHECK ---
SEQN: 0 fractional
SLD010H: 0 fraction